<a href="https://colab.research.google.com/github/keduog/EDU-AI-Training/blob/main/Day4/Session2/session2_huggingface_amharic.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Hugging Face & the Amharic Problem

**Day 4 · 10:45 – 12:45**

Find a model, find a dataset, and **prove the model is bad at Amharic** before trying to fix it.

**Runtime → Change runtime type → T4 GPU** if you have restarted.

## Cell 1 — Setup

In [ ]:
!pip install -q transformers datasets accelerate peft

import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("ready — %.2f GB used" % (torch.cuda.memory_allocated(0)/1024**3))

## Cell 2 — What is a tokenizer?

A model cannot read letters. It reads **tokens** — numbers standing for pieces of text.

Every model has its own tokenizer, and they are **not interchangeable**.

In [ ]:
sentence = "How can I track my order?"

ids    = tok.encode(sentence)
pieces = tok.tokenize(sentence)

print("Sentence :", sentence)
print("Pieces   :", pieces)
print("Numbers  :", ids)
print("Count    :", len(ids), "tokens")

## Cell 3 — The Amharic penalty

Now the same *meaning*, in Amharic. Watch the token count.

In [ ]:
pairs = [
    ("How can I track my order?",          "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?"),
    ("I want to know the price.",          "ስለ ዋጋው ማወቅ እፈልጋለሁ።"),
    ("How do I return a product?",         "ምርቱን እንዴት መመለስ እችላለሁ?"),
]

print(f"{'English':>10}   {'Amharic':>10}   ratio")
print("-" * 40)
total_en = total_am = 0
for en, am in pairs:
    n_en, n_am = len(tok.encode(en)), len(tok.encode(am))
    total_en += n_en; total_am += n_am
    print(f"{n_en:>10}   {n_am:>10}   {n_am/n_en:.1f}x")

print("-" * 40)
print(f"{total_en:>10}   {total_am:>10}   {total_am/total_en:.1f}x  OVERALL")

## Cell 4 — Look at HOW it breaks Amharic apart

In [ ]:
am = "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?"

print("Amharic pieces:")
for p in tok.tokenize(am):
    print("   ", repr(p))

### What you should notice

- Many pieces are **single characters** or even broken byte fragments
- English words often survive as whole words; Amharic rarely does
- The same meaning costs roughly **3x more tokens**

### Why this matters — and it is not just technical

Three times more tokens means:
- 3x more memory
- 3x more compute (so 3x the cost, if you are paying)
- Hitting the length limit 3x sooner

The tokenizer was trained mostly on English and Chinese text. **That was a choice someone made.**
Nobody sat down to disadvantage Amharic — but the effect is the same.

**Discuss:** if every Amharic query costs three times an English one, who pays for that?

## Cell 5 — THE BASELINE (do not skip this)

You are about to change the model. If you do not record how it behaves **now**,
you will have no way to prove anything improved later.

This is the single most-skipped step by beginners.

In [ ]:
def ask(m, question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": "You are a helpful Ethiopian customer service assistant."},
        {"role": "user",   "content": question},
    ]
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs,
                         max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


QUESTIONS = [
    "ስለ ምርታችሁ ዋጋ ማወቅ እፈልጋለሁ።",
    "ትዕዛዜን እንዴት ልከታተለው እችላለሁ?",
    "ምርቱን መመለስ ከፈለኩ ምን ማድረግ አለብኝ?",
]

before = {}
for q in QUESTIONS:
    before[q] = ask(model, q)
    print("Q:", q)
    print("A:", before[q][:200])
    print("-" * 60)

> ### Keep `before` alive for the rest of the day
> Session 4 compares against it. If Colab restarts, you must re-run this cell.

## Cell 6 — Load a real Amharic dataset

83,000 conversations, built by Ethiopian researchers, free to download.

In [ ]:
from datasets import load_dataset

DATASET = "addisai/FineTome-single-turn-dedup-amharic"

raw = load_dataset(DATASET, split="train")
print(f"{len(raw):,} examples")
print("Columns:", raw.column_names)

## Cell 7 — Always LOOK at your data first

Ten examples read carefully teach you more than 83,000 rows you never opened.

In [ ]:
import json

example = raw[0]
for key, value in example.items():
    preview = json.dumps(value, ensure_ascii=False)[:300]
    print(f"--- {key} ---")
    print(preview)
    print()

## Cell 8 — Shape it into training text

A model does not learn from a table. It learns from text showing a **question followed by a good answer**.

We wrap each example in a simple template so the model can see where the question ends
and the answer begins.

In [ ]:
def to_text(ex):
    turns = ex.get("conversations_amharic") or []
    if len(turns) < 2:
        return {"text": ""}

    question = (turns[0] or {}).get("content", "")
    answer   = (turns[1] or {}).get("content", "")

    if not question or not answer:
        return {"text": ""}

    return {"text": f"### ጥያቄ:\n{question}\n\n### መልስ:\n{answer}{tok.eos_token}"}


# 1000 examples so training finishes inside the lesson
N = 1000
data = raw.select(range(min(N, len(raw)))).map(to_text, remove_columns=raw.column_names)
data = data.filter(lambda x: len(x["text"]) > 40)

print(f"{len(data)} usable examples")
print()
print("EXAMPLE:")
print(data[0]["text"][:400])

**Why the `eos_token` at the end matters:** it marks where the answer *stops*.
Leave it out and your fine-tuned model rambles forever without ending its reply.

## Cell 9 — Save what Session 3 needs

In [ ]:
import pickle

with open("baseline.pkl", "wb") as f:
    pickle.dump({"questions": QUESTIONS, "before": before}, f)

data.save_to_disk("amharic_1000")

print("Saved: baseline.pkl and amharic_1000/")
print()
print("IMPORTANT: Colab deletes these when the session ends.")
print("Download baseline.pkl from the file panel on the left if you want to keep it.")

---

## What you learned

| Idea | Why it matters |
|---|---|
| **Tokenizers are not neutral** | Amharic costs 3x English on this model |
| **Baseline before you build** | Without a 'before', your 'after' proves nothing |
| **Look at your data** | Bad data trains bad models, silently |
| **Datasets are the hard part** | Someone spent months on those 83,000 examples |

**Next:** Session 3 — LoRA, and the actual fine-tuning run.